Here’s a clear plan for a single script on Housing.csv:

Train–test split and scaling (once)

Load Housing.csv, choose numeric features and price as target.

Do train_test_split.

Fit StandardScaler on X_train, transform both X_train and X_test for the linear, Ridge, and Lasso models.​

Fit each model on the same training data

Linear: LinearRegression()

Ridge: Ridge(alpha=...)

Lasso: Lasso(alpha=...)

Polynomial: use PolynomialFeatures(degree=2) on the original (unscaled or scaled, but consistently) features, then fit LinearRegression on X_train_poly.​

Compute the regression metrics for each
Use sklearn metrics on the test set:

mean_absolute_error(y_test, y_pred) → MAE

mean_squared_error(y_test, y_pred) → MSE

mean_squared_error(y_test, y_pred, squared=False) → RMSE

r2_score(y_test, y_pred) → R²​

In [1]:
import os
os.chdir('../Datasets')

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures


In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score

In [4]:
df = pd.read_csv('Housing.csv')
num_cols = df[df.columns[df.dtypes!='object']]

X = num_cols.drop(columns='price')
y = num_cols['price']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2)

Function to compute Matrics

In [5]:
def regression_matrix(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

results = []

In [6]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [7]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
results.append(('Linear regression', *regression_matrix(y_test, y_pred_lr)))

In [8]:

r = Ridge(alpha = 1)
r.fit(X_train_scaled, y_train)
y_pred_ridge = r.predict(X_test_scaled)
results.append(('Ridge', *regression_matrix(y_test, y_pred_ridge)))

In [9]:

l = Lasso(alpha=1)
l.fit(X_train_scaled, y_train)
y_pred_lasso = l.predict(X_test_scaled)
results.append(('Lasso', *regression_matrix(y_test, y_pred_lasso)))

In [10]:
poly = PolynomialFeatures(degree=2, include_bias = False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test)

poly_lin = LinearRegression()
poly_lin.fit(X_train_poly, y_train)
y_pred_poly = poly_lin.predict(X_test_poly)
results.append(('Poly (degree=2)', *regression_matrix(y_test, y_pred_poly)))

C:\Users\HARSH CHANDRAKAR\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but PolynomialFeatures was fitted without feature names
  warnings.warn(


In [11]:
result_cols = ["Model", "MAE", "MSE", "RMSE", "R2"]
df_results = pd.DataFrame(results, columns = result_cols)
print(df_results)

               Model           MAE           MSE          RMSE            R2
0  Linear regression  1.281699e+04  3.268600e+08  1.807927e+04  6.193434e-01
1              Ridge  1.281416e+04  3.267628e+08  1.807658e+04  6.194567e-01
2              Lasso  1.281682e+04  3.268578e+08  1.807921e+04  6.193460e-01
3    Poly (degree=2)  8.569420e+10  1.566969e+22  1.251786e+11 -1.824870e+13
